# 第 9 章 图神经网络

很多现实世界的数据天然是**图**（graph）结构的：社交网络、分子结构、知识图谱、引用网络、推荐系统……图数据具有不规则的拓扑：节点数量可变、邻居数量不定、节点之间通过边连接。前面学过的 MLP、CNN、RNN 都假设输入是**规则的**张量（向量、网格、序列），无法直接处理这种不规则结构。

**图神经网络**（Graph Neural Network，GNN）通过"在图上做消息传递"来学习节点/边/图的表示。本章实现几种最具代表性的 GNN 模型：

- **GCN**（Graph Convolutional Network，Kipf & Welling, 2017）：图卷积；
- **GraphSAGE**（Hamilton et al., 2017）：邻居采样 + 聚合；
- **GAT**（Graph Attention Network，Veličković et al., 2018）：图注意力；
- **GIN**（Graph Isomorphism Network，Xu et al., 2019）：理论表达力更强。

本章不依赖 `torch_geometric` 等第三方 GNN 库，全部用纯 PyTorch 从零实现，重点是理解消息传递的机制。

> **拓展阅读**：实际项目中推荐使用 [PyTorch Geometric (PyG)](https://pytorch-geometric.readthedocs.io/)，它提供了完整的数据集、采样器和 GNN 模型库。本章末尾给出一个 PyG 风格的实现对比。


## 1. 图数据的基本表示

一个图 $G = (V, E)$ 由节点集 $V$ 和边集 $E$ 组成。常用表示：

- **邻接矩阵** $\boldsymbol{A} \in \mathbb{R}^{N\times N}$：$A_{ij} = 1$ 当节点 $i, j$ 有边相连。
- **节点特征矩阵** $\boldsymbol{X} \in \mathbb{R}^{N\times D}$：每个节点一个 $D$ 维特征向量。
- **度矩阵** $\boldsymbol{D}$：对角矩阵，$D_{ii} = \sum_j A_{ij}$ 是节点 $i$ 的度数。

下面用一个小例子说明：

> 经典的 **Zachary's Karate Club** 数据集——一个 34 节点的社交网络，描述一个空手道俱乐部中成员的友谊关系。后来俱乐部分裂成两派（围绕教练 Mr. Hi 和管理员 Officer），我们用 GNN 学习每个成员属于哪一派。


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(0)


# ============== 构造 Karate Club 图（34 节点） ==============
# 边列表来自 Zachary 1977 原始数据
edges_raw = [
    (0,1),(0,2),(0,3),(0,4),(0,5),(0,6),(0,7),(0,8),(0,10),(0,11),
    (0,12),(0,13),(0,17),(0,19),(0,21),(0,31),
    (1,2),(1,3),(1,7),(1,13),(1,17),(1,19),(1,21),(1,30),
    (2,3),(2,7),(2,8),(2,9),(2,13),(2,27),(2,28),(2,32),
    (3,7),(3,12),(3,13),
    (4,6),(4,10),
    (5,6),(5,10),(5,16),
    (6,16),
    (8,30),(8,32),(8,33),
    (9,33),
    (13,33),
    (14,32),(14,33),
    (15,32),(15,33),
    (18,32),(18,33),
    (19,33),
    (20,32),(20,33),
    (22,32),(22,33),
    (23,25),(23,27),(23,29),(23,32),(23,33),
    (24,25),(24,27),(24,31),
    (25,31),
    (26,29),(26,33),
    (27,33),
    (28,31),(28,33),
    (29,32),(29,33),
    (30,32),(30,33),
    (31,32),(31,33),
    (32,33),
]
N = 34

# 邻接矩阵
A = torch.zeros(N, N)
for i, j in edges_raw:
    A[i, j] = 1
    A[j, i] = 1                                 # 无向图

# 度矩阵
D = A.sum(dim=1)                                 # 每个节点的度

# 节点标签：0=Mr. Hi 派系, 1=Officer 派系
y = torch.tensor([0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,1,0,0,1,0,1,0,1,1,1,1,1,1,1,1,1,1,1,1])

# 节点特征：先用 one-hot（特征数=节点数），相当于"我就是 i"，让模型自己学
X = torch.eye(N)

print(f'nodes: {N}    edges: {len(edges_raw)}    feature dim: {X.shape[1]}')
print(f'class 0 (Mr. Hi):  {(y == 0).sum().item()} nodes')
print(f'class 1 (Officer): {(y == 1).sum().item()} nodes')
print(f'avg degree: {D.mean().item():.2f}')


## 2. 消息传递机制：GNN 的核心抽象

几乎所有的 GNN 都遵循同一个**消息传递**（message passing）框架：每层 GNN 做三件事：

1. **消息计算（message）**：对每条边 $(i, j)$，从源节点 $j$ 算出要传给目标节点 $i$ 的消息 $\boldsymbol{m}_{j\to i}$；
2. **邻居聚合（aggregate）**：节点 $i$ 把来自所有邻居 $\mathcal{N}(i)$ 的消息聚合起来，比如求和或求平均；
3. **节点更新（update）**：用聚合的消息和节点自身的旧表示 $\boldsymbol{h}_i$，算出新表示 $\boldsymbol{h}'_i$。

用一个抽象公式：

$$ \boldsymbol{h}_i' = \mathrm{Update}\Big(\boldsymbol{h}_i,\ \mathrm{Aggregate}\big\{ \mathrm{Message}(\boldsymbol{h}_j,\ \boldsymbol{h}_i,\ \boldsymbol{e}_{ij})\ :\ j \in \mathcal{N}(i) \big\}\Big). $$

不同的 GNN 模型本质上就是给这三个函数选择不同的形式。下面我们用矩阵形式高效实现。


## 3. GCN：图卷积网络

Kipf & Welling 提出的 GCN 用一个紧凑的矩阵公式实现消息传递：

$$ \boldsymbol{H}^{(l+1)} = \sigma\Big(\tilde{\boldsymbol{D}}^{-\frac{1}{2}}\ \tilde{\boldsymbol{A}}\ \tilde{\boldsymbol{D}}^{-\frac{1}{2}}\ \boldsymbol{H}^{(l)}\ \boldsymbol{W}^{(l)}\Big), $$

其中：

- $\tilde{\boldsymbol{A}} = \boldsymbol{A} + \boldsymbol{I}$：加自环（让每个节点也"传消息给自己"）；
- $\tilde{\boldsymbol{D}}$：$\tilde{\boldsymbol{A}}$ 的度矩阵；
- $\tilde{\boldsymbol{D}}^{-\frac{1}{2}} \tilde{\boldsymbol{A}} \tilde{\boldsymbol{D}}^{-\frac{1}{2}}$：对称归一化的邻接矩阵，防止度数大的节点主导聚合；
- $\boldsymbol{W}^{(l)}$：可学习的权重矩阵；
- $\sigma$：非线性激活，通常 ReLU。

> 直觉：把每个节点 $i$ 的表示更新为"邻居（含自己）的表示按度数归一化后的加权和，再做线性变换 + 非线性"。


In [ ]:
def normalize_adj(A):
    """GCN 的对称归一化：A_hat = D^{-1/2} (A + I) D^{-1/2}."""
    N = A.shape[0]
    A_tilde = A + torch.eye(N)
    D_tilde = A_tilde.sum(dim=1)
    D_inv_sqrt = D_tilde.pow(-0.5)
    D_inv_sqrt[torch.isinf(D_inv_sqrt)] = 0.0
    D_mat = torch.diag(D_inv_sqrt)
    return D_mat @ A_tilde @ D_mat


class GCNLayer(nn.Module):
    """一层 GCN：H' = sigma(A_hat H W)."""

    def __init__(self, in_dim, out_dim, bias=True):
        super().__init__()
        self.lin = nn.Linear(in_dim, out_dim, bias=bias)

    def forward(self, H, A_hat):
        # H: [N, in_dim], A_hat: [N, N]
        return A_hat @ self.lin(H)


class GCN(nn.Module):
    """两层 GCN：[N, in] -> [N, hidden] -> [N, out]."""

    def __init__(self, in_dim, hidden, out_dim, dropout=0.5):
        super().__init__()
        self.conv1 = GCNLayer(in_dim, hidden)
        self.conv2 = GCNLayer(hidden, out_dim)
        self.dropout = dropout

    def forward(self, H, A_hat):
        H = F.relu(self.conv1(H, A_hat))
        H = F.dropout(H, p=self.dropout, training=self.training)
        return self.conv2(H, A_hat)


# 训练 GCN 做 Karate Club 节点分类
A_hat = normalize_adj(A)

torch.manual_seed(0)
model = GCN(in_dim=N, hidden=16, out_dim=2, dropout=0.3)
optimizer = torch.optim.Adam(model.parameters(), lr=0.05, weight_decay=5e-4)

# 半监督设定：只用 2 个节点的标签（每类 1 个）做训练
train_mask = torch.zeros(N, dtype=torch.bool)
train_mask[0]  = True       # Mr. Hi 派
train_mask[33] = True       # Officer 派

for epoch in range(200):
    model.train()
    optimizer.zero_grad()
    logits = model(X, A_hat)
    loss = F.cross_entropy(logits[train_mask], y[train_mask])
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 50 == 0:
        with torch.no_grad():
            model.eval()
            pred = model(X, A_hat).argmax(dim=1)
            acc = (pred == y).float().mean().item()
        print(f'epoch {epoch+1:3d}  loss {loss.item():.4f}  full acc {acc:.4f}')


## 4. GraphSAGE：邻居采样 + 聚合

GCN 的局限：每次更新都要算整个邻接矩阵 $\hat{\boldsymbol{A}}$，对**大图**（百万节点）显存放不下。

**GraphSAGE** 解决这个问题：把更新公式直接写成针对单个节点 $i$ 的形式，并支持**邻居采样**——每次只从 $\mathcal{N}(i)$ 里采 $K$ 个邻居参与聚合，能 scale 到任意大图。

$$ \boldsymbol{h}_i^{(l+1)} = \sigma\Big(\boldsymbol{W}^{(l)} \cdot \mathrm{CONCAT}\big(\boldsymbol{h}_i^{(l)},\ \mathrm{AGG}\{\boldsymbol{h}_j^{(l)} : j \in \mathcal{N}(i)\}\big)\Big), $$

其中 AGG 可以是 mean / sum / LSTM / max-pool。下面实现 mean aggregator 版本，并演示一层小批量训练时的邻居采样。


In [ ]:
class SAGELayer(nn.Module):
    """GraphSAGE 一层 (mean aggregator)：concat[h_i, mean(N(i))] -> linear."""

    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin = nn.Linear(2 * in_dim, out_dim)

    def forward(self, H, A):
        # H: [N, in_dim]; A: [N, N]（0/1 邻接矩阵，不需要自环）
        # 邻居均值
        deg = A.sum(dim=1, keepdim=True).clamp(min=1)
        nbr_mean = (A @ H) / deg                                # [N, in_dim]
        return self.lin(torch.cat([H, nbr_mean], dim=1))


class GraphSAGE(nn.Module):
    def __init__(self, in_dim, hidden, out_dim, dropout=0.5):
        super().__init__()
        self.conv1 = SAGELayer(in_dim, hidden)
        self.conv2 = SAGELayer(hidden, out_dim)
        self.dropout = dropout

    def forward(self, H, A):
        H = F.relu(self.conv1(H, A))
        H = F.dropout(H, p=self.dropout, training=self.training)
        return self.conv2(H, A)


torch.manual_seed(0)
model = GraphSAGE(in_dim=N, hidden=16, out_dim=2, dropout=0.3)
optimizer = torch.optim.Adam(model.parameters(), lr=0.05, weight_decay=5e-4)

for epoch in range(200):
    model.train()
    optimizer.zero_grad()
    logits = model(X, A)
    loss = F.cross_entropy(logits[train_mask], y[train_mask])
    loss.backward()
    optimizer.step()

with torch.no_grad():
    model.eval()
    pred = model(X, A).argmax(dim=1)
    acc = (pred == y).float().mean().item()
print(f'GraphSAGE  full acc {acc:.4f}')


## 5. GAT：图注意力网络

GCN 给每个邻居一样的归一化权重 $\frac{1}{\sqrt{d_i d_j}}$，但实际上有些邻居比另一些更重要——比如社交网络里，知心朋友的信息比泛泛之交更值得听。

**GAT** 用**注意力机制**让模型学习"邻居 $j$ 对节点 $i$ 有多重要"：

$$ \alpha_{ij} = \mathrm{softmax}_{j \in \mathcal{N}(i)}\Big(\mathrm{LeakyReLU}\big(\boldsymbol{a}^\top [\boldsymbol{W}\boldsymbol{h}_i \,\Vert\, \boldsymbol{W}\boldsymbol{h}_j]\big)\Big), $$

$$ \boldsymbol{h}_i' = \sigma\Big(\sum_{j \in \mathcal{N}(i) \cup \{i\}} \alpha_{ij}\ \boldsymbol{W} \boldsymbol{h}_j\Big). $$

实际中通常用**多头注意力**：跑 $K$ 套独立的 $(\boldsymbol{W}, \boldsymbol{a})$ 然后 concat。


In [ ]:
class GATLayer(nn.Module):
    """单头 GAT 层。多头通过堆多个实例 + concat 实现。"""

    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.W = nn.Linear(in_dim, out_dim, bias=False)
        self.a = nn.Parameter(torch.randn(2 * out_dim) * 0.1)
        self.leaky = nn.LeakyReLU(0.2)

    def forward(self, H, A):
        # H: [N, in_dim]; A: [N, N]; 自环
        N = H.shape[0]
        H_w = self.W(H)                                    # [N, out_dim]

        # 算所有 (i, j) 对的注意力 logits
        # e_ij = LeakyReLU(a^T [Wh_i || Wh_j])
        a_src, a_dst = self.a[:H_w.shape[1]], self.a[H_w.shape[1]:]
        e_src = H_w @ a_src                                # [N]
        e_dst = H_w @ a_dst                                # [N]
        E = e_src.unsqueeze(1) + e_dst.unsqueeze(0)        # [N, N]
        E = self.leaky(E)

        # 只在邻居 + 自环上 softmax，其他位置 mask 成 -inf
        A_self = A + torch.eye(N, device=A.device)
        E = E.masked_fill(A_self == 0, float('-inf'))
        alpha = F.softmax(E, dim=1)                        # 每行做 softmax
        return alpha @ H_w                                 # 加权聚合


class GAT(nn.Module):
    """两层多头 GAT."""

    def __init__(self, in_dim, hidden, out_dim, heads=4, dropout=0.5):
        super().__init__()
        # 第 1 层：多头，concat
        self.layer1_heads = nn.ModuleList([GATLayer(in_dim, hidden) for _ in range(heads)])
        # 第 2 层：单头输出（也可以多头取平均）
        self.layer2 = GATLayer(hidden * heads, out_dim)
        self.dropout = dropout

    def forward(self, H, A):
        H = torch.cat([head(H, A) for head in self.layer1_heads], dim=1)
        H = F.elu(H)
        H = F.dropout(H, p=self.dropout, training=self.training)
        return self.layer2(H, A)


torch.manual_seed(0)
model = GAT(in_dim=N, hidden=8, out_dim=2, heads=4, dropout=0.3)
optimizer = torch.optim.Adam(model.parameters(), lr=0.05, weight_decay=5e-4)

for epoch in range(200):
    model.train()
    optimizer.zero_grad()
    logits = model(X, A)
    loss = F.cross_entropy(logits[train_mask], y[train_mask])
    loss.backward()
    optimizer.step()

with torch.no_grad():
    model.eval()
    pred = model(X, A).argmax(dim=1)
    acc = (pred == y).float().mean().item()
print(f'GAT  full acc {acc:.4f}')


## 6. GIN：图同构网络

Xu et al. 从理论上分析了 GNN 的**表达能力**：什么样的 GNN 能区分两个不同的图？他们证明，如果 AGG 是单射函数（injective），GNN 就能达到 1-WL 图同构测试的判别能力。

GIN 用一个简单的设计实现这个：

$$ \boldsymbol{h}_i^{(l+1)} = \mathrm{MLP}^{(l)}\Big((1 + \epsilon) \cdot \boldsymbol{h}_i^{(l)} + \sum_{j \in \mathcal{N}(i)} \boldsymbol{h}_j^{(l)}\Big), $$

要点：

- 用 **sum** 而非 mean/max（sum 是单射，能区分邻居数量）；
- 用 **MLP** 而非单层 linear（MLP 可以拟合任意单射函数）；
- $\epsilon$ 可以是固定常数或可学习参数，区分中心节点的"自身贡献"。


In [ ]:
class GINLayer(nn.Module):
    def __init__(self, in_dim, out_dim, eps=0.0, train_eps=False):
        super().__init__()
        self.eps = nn.Parameter(torch.tensor(eps)) if train_eps else eps
        self.mlp = nn.Sequential(
            nn.Linear(in_dim, out_dim),
            nn.ReLU(),
            nn.Linear(out_dim, out_dim),
        )

    def forward(self, H, A):
        # 邻居 sum + (1+eps) * self
        out = (1 + self.eps) * H + A @ H
        return self.mlp(out)


class GIN(nn.Module):
    def __init__(self, in_dim, hidden, out_dim, dropout=0.5):
        super().__init__()
        self.conv1 = GINLayer(in_dim, hidden)
        self.conv2 = GINLayer(hidden, out_dim)
        self.dropout = dropout

    def forward(self, H, A):
        H = F.relu(self.conv1(H, A))
        H = F.dropout(H, p=self.dropout, training=self.training)
        return self.conv2(H, A)


torch.manual_seed(0)
model = GIN(in_dim=N, hidden=16, out_dim=2, dropout=0.3)
optimizer = torch.optim.Adam(model.parameters(), lr=0.05, weight_decay=5e-4)

for epoch in range(200):
    model.train()
    optimizer.zero_grad()
    logits = model(X, A)
    loss = F.cross_entropy(logits[train_mask], y[train_mask])
    loss.backward()
    optimizer.step()

with torch.no_grad():
    model.eval()
    pred = model(X, A).argmax(dim=1)
    acc = (pred == y).float().mean().item()
print(f'GIN  full acc {acc:.4f}')


## 7. 四种 GNN 在 Karate Club 上的对比

我们用同样的训练超参（同一个种子、200 epoch、Adam、lr=0.05、dropout=0.3）跑一遍 GCN/GraphSAGE/GAT/GIN，看在仅有 2 个标签节点的极度半监督设定下，哪种 GNN 最稳健。


In [ ]:
def train_and_eval(model_cls, model_kwargs, use_A_hat=False, n_runs=5):
    """训练多次取平均，避免单次随机性。"""
    accs = []
    for seed in range(n_runs):
        torch.manual_seed(seed)
        model = model_cls(**model_kwargs)
        opt = torch.optim.Adam(model.parameters(), lr=0.05, weight_decay=5e-4)
        adj = A_hat if use_A_hat else A
        for _ in range(200):
            model.train()
            opt.zero_grad()
            logits = model(X, adj)
            loss = F.cross_entropy(logits[train_mask], y[train_mask])
            loss.backward(); opt.step()
        with torch.no_grad():
            model.eval()
            acc = (model(X, adj).argmax(dim=1) == y).float().mean().item()
        accs.append(acc)
    return np.mean(accs), np.std(accs)


results = {}
results['GCN']       = train_and_eval(GCN,       dict(in_dim=N, hidden=16, out_dim=2, dropout=0.3), use_A_hat=True)
results['GraphSAGE'] = train_and_eval(GraphSAGE, dict(in_dim=N, hidden=16, out_dim=2, dropout=0.3))
results['GAT']       = train_and_eval(GAT,       dict(in_dim=N, hidden=8,  out_dim=2, heads=4, dropout=0.3))
results['GIN']       = train_and_eval(GIN,       dict(in_dim=N, hidden=16, out_dim=2, dropout=0.3))

print(f'{"model":<12s}{"mean acc":>10s}{"std":>10s}')
for name, (m, s) in results.items():
    print(f'{name:<12s}{m:>10.4f}{s:>10.4f}')


**典型观察**：在 Karate Club 这种**强同质性**（同社区节点更可能相连）的小图上：

- 4 种 GNN 都能达到 80%+ 的准确率；
- GCN/GIN 通常最稳定（方差小）；
- GAT 在大图上优势更明显（这里 34 节点没有体现）；
- GraphSAGE 的优势是 scale 到大图，这里同样没有体现。

> **常见误区**：把所有 GNN 当成 GCN 的变种。事实上它们的设计动机完全不同：GCN 是 spectral CNN 的近似，GraphSAGE 是面向大图的归纳学习，GAT 是 self-attention 的图变体，GIN 是表达力理论驱动。


## 8. 学到的节点嵌入

把第一层 GCN 输出（隐藏层）用 PCA 降到 2 维，看一下学到的节点表示是否能在空间上自然分开两个派系。


In [ ]:
torch.manual_seed(0)
gcn = GCN(in_dim=N, hidden=16, out_dim=2, dropout=0.0)
opt = torch.optim.Adam(gcn.parameters(), lr=0.05, weight_decay=5e-4)
for _ in range(200):
    gcn.train()
    opt.zero_grad()
    loss = F.cross_entropy(gcn(X, A_hat)[train_mask], y[train_mask])
    loss.backward(); opt.step()

# 取出第一层的输出作为节点嵌入
with torch.no_grad():
    gcn.eval()
    h1 = gcn.conv1(X, A_hat).numpy()

# PCA 到 2 维
from sklearn.decomposition import PCA
emb_2d = PCA(n_components=2).fit_transform(h1)

plt.figure(figsize=(7, 6))
colors = ['tab:blue', 'tab:red']
for k in [0, 1]:
    plt.scatter(emb_2d[y.numpy() == k, 0], emb_2d[y.numpy() == k, 1],
                s=120, alpha=0.7, label=f'class {k}', c=colors[k])
for i in range(N):
    plt.annotate(str(i), (emb_2d[i, 0], emb_2d[i, 1]), fontsize=7, ha='center', va='center')
plt.title('GCN node embeddings (PCA-2D)')
plt.legend(); plt.grid(alpha=0.3); plt.show()


## 9. 玩具案例：分子图回归

GNN 的一个重要应用是**图级任务**（不是节点级）：给一个完整的图，预测一个标量或类别。典型场景是分子性质预测——给一个分子的化学结构图，预测其能量、毒性、溶解度等。

我们造一个超简化的玩具数据集：每个图随机生成 5～10 个节点的小图，每个节点有 3 维特征。**任务**：预测整图的"密度分数" = 边数 / 节点数 × 节点特征之和的某个统计量。

这一节的重点不是结果好坏，而是演示 GNN 在图级任务上的**读出**（readout）操作：把所有节点表示聚合成一个图级表示。


In [ ]:
def random_graph(min_n=5, max_n=10, feat_dim=3, edge_prob=0.4, seed=None):
    """随机生成一个小图（Erdős–Rényi 模型）。"""
    g = torch.Generator()
    if seed is not None:
        g.manual_seed(seed)
    n = torch.randint(min_n, max_n + 1, (1,), generator=g).item()
    Xg = torch.randn(n, feat_dim, generator=g)
    Ag = (torch.rand(n, n, generator=g) < edge_prob).float()
    Ag = (Ag + Ag.t()) / 2
    Ag = (Ag > 0).float()
    Ag.fill_diagonal_(0)
    n_edges = (Ag.sum() / 2).item()
    target = n_edges / n + 0.3 * Xg.sum().item() / n
    return Xg, Ag, torch.tensor(target, dtype=torch.float32)


# 生成数据集
torch.manual_seed(0)
train_data = [random_graph(seed=i) for i in range(200)]
test_data  = [random_graph(seed=1000 + i) for i in range(50)]
print(f'train {len(train_data)} graphs    test {len(test_data)} graphs')


# ============== 图级 GIN：多层消息传递 + 全图 readout ==============
class GraphGIN(nn.Module):
    def __init__(self, in_dim, hidden, n_layers=3, dropout=0.0):
        super().__init__()
        self.convs = nn.ModuleList()
        for i in range(n_layers):
            d_in = in_dim if i == 0 else hidden
            self.convs.append(GINLayer(d_in, hidden))
        # 把每一层 readout 后的 hidden 表示拼起来再回归
        self.lin = nn.Linear(hidden * n_layers, 1)
        self.dropout = dropout

    def forward(self, H, A):
        outs = []
        for conv in self.convs:
            H = F.relu(conv(H, A))
            outs.append(H.sum(dim=0))                    # sum readout: [hidden]
        h_graph = torch.cat(outs, dim=0)                 # [hidden * n_layers]
        return self.lin(F.dropout(h_graph, p=self.dropout, training=self.training)).squeeze()


torch.manual_seed(0)
model = GraphGIN(in_dim=3, hidden=16, n_layers=3, dropout=0.2)
opt = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.MSELoss()

# 一个简单的"逐图"训练循环（小数据集，直接 for 循环够用）
for epoch in range(50):
    model.train()
    total = 0.0
    for Xg, Ag, target in train_data:
        opt.zero_grad()
        pred = model(Xg, Ag)
        loss = loss_fn(pred, target)
        loss.backward(); opt.step()
        total += loss.item()
    if (epoch + 1) % 10 == 0:
        with torch.no_grad():
            model.eval()
            test_mse = sum(loss_fn(model(Xg, Ag), t).item() for Xg, Ag, t in test_data) / len(test_data)
        print(f'epoch {epoch+1:2d}  train mse {total/len(train_data):.4f}  test mse {test_mse:.4f}')


**关键观察**：

- 在 `GraphGIN` 里我们用了 **sum readout**（把所有节点的隐藏表示求和），这样图级输出对节点数量是敏感的；
- 我们还**拼接了每一层的 readout**，这是 GIN 原论文推荐的做法——不同深度的层捕捉不同尺度的子结构信息；
- 对真实数据（如 OGB Molecular），通常配合 mini-batch 训练。下面的"小批量训练 tips"给出实战要点。


## 10. 拓展：PyTorch Geometric 等价实现

实际项目中，[PyTorch Geometric](https://pytorch-geometric.readthedocs.io/) 提供了更完整、更高效的 GNN 实现，支持：

- 各种图数据集（Cora、Citeseer、PubMed、OGB、QM9 等）的一键加载；
- 高效的稀疏邻接表表示（`edge_index`）；
- 邻居采样、mini-batch、异构图等；
- 50+ 种 GNN 层（`GCNConv`、`SAGEConv`、`GATConv`、`GINConv` 等）。

如果安装了 PyG，等价的 GCN 写法是：

```python
import torch_geometric as pyg
from torch_geometric.nn import GCNConv

class GCN_PyG(torch.nn.Module):
    def __init__(self, in_dim, hidden, out_dim):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hidden)
        self.conv2 = GCNConv(hidden, out_dim)
    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=0.5, training=self.training)
        return self.conv2(x, edge_index)
```

PyG 用 `edge_index` `[2, E]` 替代邻接矩阵 `[N, N]`，对稀疏大图（百万节点、亿条边）更省内存。安装：

```bash
pip install torch_geometric
```

> 安装可能需要额外的编译扩展（`torch_scatter`, `torch_sparse`），具体看 PyG 官方文档。


## 小结

本章实现了 4 种主流 GNN：

| 模型 | 关键思路 | 公式核心 |
|---|---|---|
| **GCN** | 对称归一化的图卷积 | $\tilde{D}^{-1/2}\tilde{A}\tilde{D}^{-1/2} H W$ |
| **GraphSAGE** | 邻居采样 + 聚合 | $\sigma(W[h_i\,\Vert\,\mathrm{AGG}(\mathcal{N}(i))])$ |
| **GAT** | 学习邻居权重 | $\sum_j \alpha_{ij} W h_j$ |
| **GIN** | 单射聚合 + MLP | $\mathrm{MLP}((1+\epsilon)h_i + \sum_j h_j)$ |

**实战经验速查**：

- **节点级任务**（Cora、社交网络）通常用 GCN 或 GAT；强同质图用 GCN 足够，弱同质或异质图考虑 GAT/GraphSAGE。
- **图级任务**（分子性质预测）通常用 GIN，因为它表达力理论保证最好；记得加多层 readout。
- **大图**（百万+节点）必须用 GraphSAGE 这类邻居采样方法，或者考虑 Cluster-GCN/GraphSAINT 这类图采样。
- **过度平滑问题**：层数太深时所有节点表示趋同。一般保持 2-3 层；用残差连接、跳跃连接缓解。

下一章我们会进入**大语言模型与智能体**——把注意力机制扩展到序列上的 Transformer 模型，并构建一个简单的智能体。


<!-- MIGRATED-FROM-BOOK: 图神经网络 补充小节 -->
## 补充小节：从书里迁移的进阶内容

以下小节是配套教材《案例与实践2·图神经网络》里出现、而前面 notebook 尚未覆盖的内容，按主题补齐：

1. **消息传递抽象基类** `MessagePassing` 与 `MeanConv` 示例（纯 PyTorch，可直接运行）；
2. **过度平滑**：`DeepGCN` / `node_similarity` / `ResGCN` 残差缓解（合成小图与 Karate 图，可运行）；
3. **链路预测**：负采样、切边、`GCNEncoder` + 内积解码器、ROC 评估（可运行）；
4. **Karate Club 拓扑 + 邻接矩阵热力图**可视化（`networkx`，可运行）；
5. **Cora 实战**（`Planetoid` 加载、`GCN_PyG`、t-SNE、Label Propagation 基线）——需 `torch_geometric`，仅给可运行代码，配套 venv 已跑通；新环境需先安装 PyG；
6. **图分类 PROTEINS**（`TUDataset` / `DataLoader` / `GINConv` + `global_add_pool`）——需 `torch_geometric`，同上。

> 这些 cell 直接复用前面已定义的 `N`、`A`、`X`、`y`、`A_hat`、`train_mask`、`normalize_adj`、`GCNLayer`、`GINLayer` 等变量与类，请在运行完前面的小节后再依次执行。

### 补充 1：Karate Club 拓扑与邻接矩阵可视化

GNN 的“可解释性”很大程度上靠可视化建立：先把图画出来、把邻接矩阵画出来，最后再把学到的表示画出来（节点嵌入散点见前面第 8 节）。这里先观察图的拓扑——节点按真实派别着色，用 `networkx` 的 `spring_layout` 布局画出社交网络，再把邻接矩阵 `A` 画成热力图。

前面 notebook 是用原始边列表手工构造邻接矩阵 `A` 的，这里直接用 `nx.from_numpy_array(A)` 把同一张图交给 `networkx` 绘制，保证节点编号与标签 `y` 完全一致。

In [ ]:
import networkx as nx

# 用前面已构造的邻接矩阵 A 还原成 networkx 图（节点编号、标签都与 y 对齐）
G_nx = nx.from_numpy_array(A.numpy())
pos = nx.spring_layout(G_nx, seed=42)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))

# 左：图拓扑，颜色=真实派别
node_colors = ['#4C72B0' if int(yi) == 0 else '#DD8452' for yi in y]
nx.draw_networkx(
    G_nx, pos, ax=axes[0],
    node_color=node_colors, node_size=320,
    edgecolors='white', linewidths=1.0,
    font_size=8, font_color='white',
    edge_color='#BBBBBB', width=1.0,
)
axes[0].set_title('Karate Club 图拓扑（蓝=Mr. Hi 派，橙=Officer 派）')
axes[0].axis('off')

# 右：邻接矩阵热力图
im = axes[1].imshow(A.numpy(), cmap='Blues')
axes[1].set_title('邻接矩阵热力图')
axes[1].set_xlabel('节点 j')
axes[1].set_ylabel('节点 i')
fig.colorbar(im, ax=axes[1], fraction=0.046)

plt.tight_layout()
plt.show()

print(f'节点数 N={N}，边数 |E|={len(edges_raw)}，平均度数={2*len(edges_raw)/N:.2f}')
print(f'度数分布：min={int(D.min())}，max={int(D.max())}，mean={D.mean():.2f}')

从可视化里可以读出三个直观规律：

- 网络呈现明显的“两团”结构——同派内部连接紧密、跨派连接稀疏，这正是节点分类背后的**同质性假设**（homophily）；
- 邻接矩阵接近**块对角**：左上块（Mr. Hi 派）和右下块（Officer 派）密度高，跨块（左下／右上）稀疏；
- 矩阵关于对角线对称，因为这是一张**无向图**。

### 补充 2：消息传递抽象基类与 MeanConv 示例

前面的 GCN/GraphSAGE/GAT/GIN 都是直接用矩阵形式写的。为了把**消息传递**这个统一范式独立出来，下面实现一个轻量级的抽象基类 `MessagePassing`：子类只需重写 `message`、`aggregate`、`update` 三个钩子，框架负责把它们按“消息计算 → 邻居聚合 → 节点更新”的顺序串起来。这与 PyTorch Geometric 的 `torch_geometric.nn.MessagePassing` 思路相同，区别在于去掉了对 `edge_index` 稀疏求和的工程优化，便于阅读。

> 为了清晰，这里用 `[N, N, D]` 的稠密张量来表达每条边的消息——在 Karate Club 这种小图上没问题，但在百万节点的大图上会爆显存。PyG 实际是基于 `edge_index` 做稀疏 `scatter_` 聚合的，原理一致、规模一致，只是内存友好得多。

In [ ]:
class MessagePassing(nn.Module):
    """消息传递抽象基类。

    Args:
        aggr (str): 聚合方式，可选 'sum' / 'mean' / 'max'。
    """

    def __init__(self, aggr='sum'):
        super().__init__()
        assert aggr in ('sum', 'mean', 'max')
        self.aggr = aggr

    # —— 三个钩子函数，子类按需重写 ——————————————————————
    def message(self, h_src, h_dst, edge_attr=None):
        """计算每条边从 src 流向 dst 的消息。默认直接传 h_src。"""
        return h_src

    def aggregate(self, messages, A):
        """把 N x N x D 的边消息按 dst 聚合成 N x D 的节点消息。"""
        # messages: [N, N, D]，A: [N, N]
        if self.aggr == 'sum':
            return torch.einsum('ij,ijd->id', A, messages)
        if self.aggr == 'mean':
            deg = A.sum(dim=1, keepdim=True).clamp(min=1)
            return torch.einsum('ij,ijd->id', A, messages) / deg
        # max
        m = messages.clone()
        m[A == 0] = float('-inf')
        return m.max(dim=1).values.clamp(min=-1e9)

    def update(self, h_old, h_agg):
        """用聚合后的消息更新节点表示。默认直接返回 h_agg。"""
        return h_agg

    # —— 统一 forward：子类一般不需要重写 ————————————————
    def forward(self, H, A, edge_attr=None):
        # 广播成 [N, N, D] 的边消息张量
        n, d = H.shape
        h_src = H.unsqueeze(0).expand(n, n, d)        # 沿 dst 维广播
        h_dst = H.unsqueeze(1).expand(n, n, d)        # 沿 src 维广播
        msg = self.message(h_src, h_dst, edge_attr)
        agg = self.aggregate(msg, A)
        return self.update(H, agg)


class MeanConv(MessagePassing):
    """h_i' = W [ h_i || mean(N(i)) ]，与 GraphSAGE 的 mean 变体几乎一致。"""

    def __init__(self, in_dim, out_dim):
        super().__init__(aggr='mean')
        self.lin = nn.Linear(2 * in_dim, out_dim)

    def update(self, h_old, h_agg):
        return self.lin(torch.cat([h_old, h_agg], dim=1))


# 测试：随便造一个 5 节点的小图
A_demo = torch.tensor([
    [0, 1, 1, 0, 0],
    [1, 0, 1, 0, 0],
    [1, 1, 0, 1, 0],
    [0, 0, 1, 0, 1],
    [0, 0, 0, 1, 0],
], dtype=torch.float32)
H_demo = torch.randn(5, 4)
out = MeanConv(4, 8)(H_demo, A_demo)
print('MeanConv 输出 shape:', tuple(out.shape))        # (5, 8)

把 GCN、GraphSAGE、GAT、GIN 都套进这个框架，差异可以简洁地概括为：

- **GCN**：message $=\frac{1}{\sqrt{d_i d_j}}\boldsymbol{W}\boldsymbol{h}_j$，aggr $=$ sum，update $=$ identity；
- **GraphSAGE**：message $=\boldsymbol{h}_j$，aggr $=$ mean，update $=\boldsymbol{W}[\boldsymbol{h}_i\,\Vert\,\boldsymbol{m}_i]$；
- **GAT**：message $=\alpha_{ij}\boldsymbol{W}\boldsymbol{h}_j$（注意力加权），aggr $=$ sum，update $=$ identity；
- **GIN**：message $=\boldsymbol{h}_j$，aggr $=$ sum，update $=\mathrm{MLP}((1+\epsilon)\boldsymbol{h}_i + \boldsymbol{m}_i)$。

### 补充 3：一个通用训练循环 `train_gnn`

后面的过度平滑实验会反复训练不同深度的 GCN，先把训练流程抽成一个通用函数（返回每个 epoch 的 loss 和全图准确率），方便复用。它直接使用前面定义好的 `X`、`y`、`train_mask`。

In [ ]:
def train_gnn(model, A_input, n_epoch=200, lr=0.05, wd=5e-4, seed=0):
    """通用训练循环：返回每个 epoch 的 (loss, acc)。"""
    torch.manual_seed(seed)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    hist = {'loss': [], 'acc': []}
    for _ in range(n_epoch):
        model.train()
        opt.zero_grad()
        logits = model(X, A_input)
        loss = F.cross_entropy(logits[train_mask], y[train_mask])
        loss.backward()
        opt.step()
        with torch.no_grad():
            model.eval()
            acc = (model(X, A_input).argmax(1) == y).float().mean().item()
            hist['loss'].append(loss.item())
            hist['acc'].append(acc)
    return hist


# 快速自检：复跑两层 GCN，应当 ~0.97
_h = train_gnn(GCN(N, 16, 2, dropout=0.3), A_hat, n_epoch=200)
print(f'train_gnn 自检：两层 GCN full acc = {_h["acc"][-1]:.4f}')

### 补充 4：过度平滑——层数为什么不能堆太多

CNN 与 Transformer 都通过堆深度获取更大的感受野；而 GNN 直到今天主流仍只用 2 到 3 层。原因是一种 GNN 特有的现象——**过度平滑**（over-smoothing）：层数过深时，所有节点的表示迅速“趋同”，可分性随之消失。

直觉上，每层 GCN 都把节点表示朝邻居均值的方向拉一步，重复足够多次后所有节点都会收敛到同一个不动点（对应 $\hat{\boldsymbol{A}}$ 最大特征向量方向）。下面把 GCN 堆到 2、4、6、8 层做一组对比，并用**节点表示两两余弦相似度的均值** `node_similarity` 作为过度平滑的量化指标。

In [ ]:
class DeepGCN(nn.Module):
    """L 层 GCN。"""

    def __init__(self, in_dim, hidden, out_dim, n_layers=2, dropout=0.5):
        super().__init__()
        dims = [in_dim] + [hidden] * (n_layers - 1) + [out_dim]
        self.layers = nn.ModuleList([GCNLayer(dims[i], dims[i + 1])
                                     for i in range(n_layers)])
        self.dropout = dropout

    def forward(self, H, A_hat):
        for i, layer in enumerate(self.layers):
            H = layer(H, A_hat)
            if i < len(self.layers) - 1:
                H = F.relu(H)
                H = F.dropout(H, p=self.dropout, training=self.training)
        return H


def node_similarity(H):
    """节点表示两两余弦相似度的均值——衡量过度平滑的指标。"""
    H = F.normalize(H, dim=1)
    return (H @ H.t()).mean().item()


depths = [2, 4, 6, 8]
res = {}
for L in depths:
    torch.manual_seed(0)
    m = DeepGCN(N, 16, 2, n_layers=L, dropout=0.3)
    h = train_gnn(m, A_hat, n_epoch=200)
    m.eval()
    with torch.no_grad():
        # 取倒数第二层（hidden 维）的表示做相似度统计
        H_hidden = X
        for layer in m.layers[:-1]:
            H_hidden = F.relu(layer(H_hidden, A_hat))
    res[L] = {'acc': h['acc'][-1], 'sim': node_similarity(H_hidden)}
    print(f'L={L}  acc={res[L]["acc"]:.4f}  cosine={res[L]["sim"]:.4f}')

In [ ]:
# 把“层数 vs 准确率 / 节点相似度”画出来
Ls = list(res.keys())
accs = [res[L]['acc'] for L in Ls]
sims = [res[L]['sim'] for L in Ls]

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].plot(Ls, accs, 'o-', color='#4C72B0')
ax[0].set_xlabel('层数 L')
ax[0].set_ylabel('全图准确率')
ax[0].set_title('层数越深，准确率越低')
ax[0].grid(alpha=0.3)

ax[1].plot(Ls, sims, 's-', color='#DD8452')
ax[1].set_xlabel('层数 L')
ax[1].set_ylabel('节点表示余弦相似度')
ax[1].set_title('层数越深，节点越“变得一样”')
ax[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

两个观察：

- **准确率单调下降**：从 2 层的约 0.97 掉到 8 层附近——深层模型几乎只是“瞎猜”；
- **节点表示相似度单调上升**：从 2 层的约 0.4 上升到 8 层接近 1.0——所有节点的表示几乎重合，分类器无法区分。

缓解过度平滑主流有三类思路：**残差连接**（ResGCN／JK-Net，让深层至少不比浅层差）、**表示归一化**（PairNorm／NodeNorm，每层后把节点表示重新拉开）、**图稀疏化**（DropEdge，随机丢边降低扩散速度）。下面实现带残差连接的 `ResGCN`。

In [ ]:
class ResGCN(nn.Module):
    """带残差连接的 L 层 GCN。"""

    def __init__(self, in_dim, hidden, out_dim, n_layers=6, dropout=0.3):
        super().__init__()
        self.in_lin = nn.Linear(in_dim, hidden)
        self.layers = nn.ModuleList([GCNLayer(hidden, hidden)
                                     for _ in range(n_layers - 1)])
        self.out_lin = nn.Linear(hidden, out_dim)
        self.dropout = dropout

    def forward(self, H, A_hat):
        H = F.relu(self.in_lin(H))
        for layer in self.layers:
            H = F.relu(layer(H, A_hat)) + H                  # 残差
            H = F.dropout(H, p=self.dropout, training=self.training)
        return self.out_lin(H)


torch.manual_seed(0)
h_res = train_gnn(ResGCN(N, 16, 2, n_layers=6), A_hat, n_epoch=200)
print(f'ResGCN-6L acc={h_res["acc"][-1]:.4f}')        # 残差让 6 层恢复到接近浅层水平

> **注意**：GNN 里的“深”和 CNN 里的“深”量级完全不同。CNN 常见 100 层以上，而 GNN 即使加了残差，多数论文也只堆到 4 到 6 层。原因是 GCN 的感受野**随层数指数式增长**——4 层基本就能覆盖整张图，再深就是冗余 + 过度平滑。

### 补充 5：链路预测——让模型“补全”图

到此为止所有任务都属于**节点分类**。GNN 还有另一类经典任务——**链路预测**（link prediction）：基于图的一部分已知边，预测两个节点之间**是否本就连着**（或将来是否会形成）一条边。社交网络的“可能认识的人”、知识图谱补全、推荐系统的 user–item 二部图都属于这类。

主流范式是“编码器–解码器”：

$$\boldsymbol{z}_i = \mathrm{GNN}(\boldsymbol{X}, \boldsymbol{A})_i \quad (\text{encoder}), \qquad \hat{p}_{ij} = \mathrm{sigmoid}(\boldsymbol{z}_i^\top \boldsymbol{z}_j) \quad (\text{decoder}).$$

encoder 用 GCN 把每个节点编成 $d$ 维向量；decoder 用最简单的**内积**评分判断“$i$ 和 $j$ 之间应该有边”的概率。训练目标是让真实边得分高、不存在的边（负样本）得分低，本质是一个二分类问题。

先写两个工具：**负采样**（从“缺边”里造负样本）和**切分训练／测试边**（从原图里挖掉一部分真实边作测试集）。

In [ ]:
def sample_neg_edges(A, n_neg, exclude=None):
    """从邻接矩阵 A 里采 n_neg 条不存在的边作为负样本。"""
    n = A.shape[0]
    if exclude is None:
        exclude = A.bool()
    neg = []
    while len(neg) < n_neg:
        i = np.random.randint(n)
        j = np.random.randint(n)
        if i != j and not exclude[i, j] and (i, j) not in neg:
            neg.append((i, j))
    return torch.tensor(neg).t()                 # [2, n_neg]


def split_edges(A, test_ratio=0.1, seed=0):
    """把无向图的边按比例切成训练 / 测试，返回训练邻接 + 训练边 + 测试边。"""
    np.random.seed(seed)
    n = A.shape[0]
    edges = [(i, j) for i in range(n) for j in range(i + 1, n) if A[i, j] > 0]
    np.random.shuffle(edges)
    n_test = int(len(edges) * test_ratio)
    test_edges = edges[:n_test]
    train_edges = edges[n_test:]

    A_train = torch.zeros_like(A)
    for i, j in train_edges:
        A_train[i, j] = 1
        A_train[j, i] = 1
    return A_train, torch.tensor(train_edges).t(), torch.tensor(test_edges).t()


A_train, train_pos, test_pos = split_edges(A, test_ratio=0.1, seed=0)
print(f'训练边 {train_pos.shape[1]} / 测试边 {test_pos.shape[1]}')

下面用前面的 `GCNLayer` 搭一个 `GCNEncoder`，配内积 decoder，做完整的链路预测训练。评估指标用 **AUC**（ROC 曲线下面积）——对链路预测来说准确率不合适，因为大多数“边”其实不存在，全预测“不存在”就能拿到接近 1 的准确率。

In [ ]:
class GCNEncoder(nn.Module):
    """GCN encoder：输出每个节点的 d 维 embedding。"""

    def __init__(self, in_dim, hidden, out_dim):
        super().__init__()
        self.conv1 = GCNLayer(in_dim, hidden)
        self.conv2 = GCNLayer(hidden, out_dim)

    def forward(self, H, A_hat):
        H = F.relu(self.conv1(H, A_hat))
        return self.conv2(H, A_hat)


def decode(Z, edge_index):
    """对一组边 (i, j) 做内积评分。"""
    return (Z[edge_index[0]] * Z[edge_index[1]]).sum(dim=1)


# —— 训练 ——————————————————————————————————————————————
from sklearn.metrics import roc_auc_score

torch.manual_seed(0)
np.random.seed(0)
A_hat_train = normalize_adj(A_train)
encoder = GCNEncoder(N, 32, 16)
opt = torch.optim.Adam(encoder.parameters(), lr=0.01, weight_decay=5e-4)

train_loss_hist, test_auc_hist = [], []
labels = scores = None
for epoch in range(200):
    encoder.train()
    opt.zero_grad()
    Z = encoder(X, A_hat_train)
    # 正样本得分
    pos_score = decode(Z, train_pos)
    # 负样本得分（每个 epoch 重新采）
    neg_idx = sample_neg_edges(A_train, n_neg=train_pos.shape[1])
    neg_score = decode(Z, neg_idx)
    # 二分类 logistic loss
    pos_loss = F.binary_cross_entropy_with_logits(pos_score, torch.ones_like(pos_score))
    neg_loss = F.binary_cross_entropy_with_logits(neg_score, torch.zeros_like(neg_score))
    loss = pos_loss + neg_loss
    loss.backward()
    opt.step()
    train_loss_hist.append(loss.item())
    # 测试 AUC（每 10 epoch）
    if (epoch + 1) % 10 == 0:
        encoder.eval()
        with torch.no_grad():
            Z = encoder(X, A_hat_train)
            test_neg = sample_neg_edges(A, n_neg=test_pos.shape[1])
            pos = decode(Z, test_pos).sigmoid().numpy()
            neg = decode(Z, test_neg).sigmoid().numpy()
        labels = np.concatenate([np.ones_like(pos), np.zeros_like(neg)])
        scores = np.concatenate([pos, neg])
        auc = roc_auc_score(labels, scores)
        test_auc_hist.append(auc)
        if (epoch + 1) % 50 == 0:
            print(f'epoch {epoch+1:3d}  loss={loss.item():.4f}  test_AUC={auc:.4f}')

把训练损失和 ROC 曲线画出来。Karate Club 这种小图上测试边只有几条，AUC 会有一定随机波动，但通常能到 0.8 以上，远高于随机猜测的 0.5。

> 链路预测的 encoder 和 decoder 是**解耦**的，可以自由组合：encoder 换成 GraphSAGE／GAT 等任意节点表示模型；decoder 除内积外还有 DistMult、ComplEx（知识图谱）和双线性层 $\boldsymbol{z}_i^\top \boldsymbol{W} \boldsymbol{z}_j$。“GCN + 内积”组合最早出现在 2016 年的 VGAE 论文里，至今仍是链路预测最常用的 baseline。

In [ ]:
from sklearn.metrics import roc_curve

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].plot(train_loss_hist, color='#FF6B6B')
ax[0].set_title('训练 Loss')
ax[0].set_xlabel('epoch')
ax[0].grid(alpha=0.3)

fpr, tpr, _ = roc_curve(labels, scores)
ax[1].plot(fpr, tpr, color='#4ECDC4', lw=2, label=f'AUC = {auc:.3f}')
ax[1].plot([0, 1], [0, 1], 'k--', alpha=0.4)
ax[1].set_xlabel('False Positive Rate')
ax[1].set_ylabel('True Positive Rate')
ax[1].set_title('链路预测 ROC 曲线')
ax[1].legend()
ax[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

### 补充 6：真实数据集实战——Cora 论文引用网络（需 `torch_geometric`）

**Cora** 是节点分类的“Hello World”：2708 个节点（论文），每篇论文由 1433 维词袋向量表示；10556 条无向引用边；7 个类别；标准切分为 140 个训练标签（每类 20 个）、500 验证、1000 测试。这 140 个训练标签只占 5.2%——仍是半监督设定，但比 Karate Club 的“2 个标签”贴近真实场景。

> **运行环境说明**：以下两节（Cora、PROTEINS）依赖 `torch_geometric`（PyG）。当前配套 venv 已经跑通；如果换到新环境，请先按 PyG 官方文档安装对应版本。首次运行会自动下载数据集。

In [ ]:
# 需 torch_geometric；首次运行会自动下载数据集。
from torch_geometric.datasets import Planetoid
from torch_geometric.transforms import NormalizeFeatures
from torch_geometric.nn import GCNConv

dataset = Planetoid(root='data/Planetoid', name='Cora',
                    transform=NormalizeFeatures())
data = dataset[0]
print(data)
print(f'类别数 = {dataset.num_classes}, 特征维数 = {dataset.num_features}')
print(f'训练/验证/测试 = {data.train_mask.sum()} / '
      f'{data.val_mask.sum()} / {data.test_mask.sum()}')


class GCN_PyG(nn.Module):
    def __init__(self, in_dim, hidden, out_dim, dropout=0.5):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hidden)
        self.conv2 = GCNConv(hidden, out_dim)
        self.dropout = dropout

    def forward(self, x, edge_index):
        # 注意接口变化：第二个参数是 edge_index（稀疏边表）而非 A_hat。
        # GCNConv 内部按需做对称归一化，不用手动算 D^{-1/2}。
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        return self.conv2(x, edge_index)

In [ ]:
# 训练 GCN（完整流程：train + 用 val 选 best 模型）—— 需 torch_geometric
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
data = data.to(device)
model = GCN_PyG(in_dim=dataset.num_features, hidden=16,
                out_dim=dataset.num_classes, dropout=0.5).to(device)
opt = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

best_val_acc = 0.0
best_state = None
history = {'train_loss': [], 'val_acc': [], 'test_acc': []}
for epoch in range(1, 201):
    model.train()
    opt.zero_grad()
    out = model(data.x, data.edge_index)
    loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    opt.step()

    model.eval()
    with torch.no_grad():
        pred = model(data.x, data.edge_index).argmax(dim=1)
        val_acc = (pred[data.val_mask] == data.y[data.val_mask]).float().mean().item()
        test_acc = (pred[data.test_mask] == data.y[data.test_mask]).float().mean().item()
    history['train_loss'].append(loss.item())
    history['val_acc'].append(val_acc)
    history['test_acc'].append(test_acc)
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
    if epoch % 20 == 0:
        print(f'epoch {epoch:3d}  loss={loss.item():.4f}  '
              f'val={val_acc:.4f}  test={test_acc:.4f}')

model.load_state_dict(best_state)
model.eval()
with torch.no_grad():
    pred = model(data.x, data.edge_index).argmax(dim=1)
    final_test = (pred[data.test_mask] == data.y[data.test_mask]).float().mean().item()
print(f'\nBest-val 模型 test_acc = {final_test:.4f}')        # 典型 0.81

In [ ]:
# 结果分析：t-SNE 可视化 7 个类别 —— 需 torch_geometric + sklearn
from sklearn.manifold import TSNE

model.eval()
with torch.no_grad():
    h = F.relu(model.conv1(data.x, data.edge_index)).cpu().numpy()
emb = TSNE(n_components=2, perplexity=30, random_state=0).fit_transform(h)

labels = data.y.cpu().numpy()
class_names = ['Neural Networks', 'RL', 'Prob. Methods', 'Rule Learning',
               'Genetic Alg.', 'Theory', 'Case Based']
palette = plt.cm.tab10.colors
plt.figure(figsize=(7, 6))
for c in range(7):
    plt.scatter(emb[labels == c, 0], emb[labels == c, 1],
                c=[palette[c]], s=8, label=class_names[c], alpha=0.7)
plt.legend(loc='best', fontsize=8)
plt.title('Cora 上 GCN 隐层 t-SNE')
plt.xticks([])
plt.yticks([])
plt.tight_layout()
plt.show()

为了说明“GNN 为什么有用”，把 GCN 和两个不用图结构的基线比一下：**MLP**（只用节点特征 $\boldsymbol{X}$，完全无视边）和 **Label Propagation**（完全不用特征，只靠图结构传标签）。

| 模型 | Test Acc | 用了什么信息 |
|---|---|---|
| MLP（只用特征） | ~0.58 | 只用 $\boldsymbol{X}$，不看图 |
| Label Propagation | ~0.68 | 只用 $\boldsymbol{A}$，不看 $\boldsymbol{X}$ |
| **GCN（2 层）** | **~0.81** | 同时用 $\boldsymbol{A}$ 和 $\boldsymbol{X}$ |
| GAT（2 层 8 头） | ~0.83 | 同上 + 注意力 |

结论清晰：图结构和节点特征都缺一不可，把它们结合起来的 GNN 比单独使用任何一个都好不少。

In [ ]:
# 基线对比：MLP（只用特征）与 Label Propagation（只用图结构）—— 需 torch_geometric
class MLP(nn.Module):
    def __init__(self, in_dim, hidden, out_dim, dropout=0.5):
        super().__init__()
        self.lin1 = nn.Linear(in_dim, hidden)
        self.lin2 = nn.Linear(hidden, out_dim)
        self.dropout = dropout

    def forward(self, x, edge_index=None):                    # 接口对齐 GCN_PyG
        return self.lin2(F.dropout(F.relu(self.lin1(x)),
                                   p=self.dropout, training=self.training))


# Label Propagation 基线：PyG 自带，无需训练，只用图结构传标签
from torch_geometric.nn import LabelPropagation

lp = LabelPropagation(num_layers=3, alpha=0.9)
lp_out = lp(data.y, data.edge_index, mask=data.train_mask)
lp_pred = lp_out.argmax(dim=1)
lp_acc = (lp_pred[data.test_mask] == data.y[data.test_mask]).float().mean().item()
print(f'Label Propagation test_acc = {lp_acc:.4f}')          # 典型 ~0.68

#### 模型预测

最后用训练好的模型对单个测试节点做一次预测，直观感受“给定一篇论文，模型判断它属于哪个研究方向”。

In [1]:
model.eval()
with torch.no_grad():
    probs = F.softmax(model(data.x, data.edge_index), dim=1)
# 任取一个测试集中的节点
node = data.test_mask.nonzero(as_tuple=True)[0][0].item()
pred = probs[node].argmax().item()
print(f'节点 {node}：预测 = {class_names[pred]}'
      f'（置信度 {probs[node, pred]:.2f}），真实 = {class_names[data.y[node].item()]}')

节点 1708：预测 = RL（置信度 0.19），真实 = Rule Learning


### 补充 7：图分类实战——分子属性预测 PROTEINS（需 `torch_geometric`）

前面所有任务都在**一张图**上做节点级任务。化学、药物、生物领域则常常面对**大量小图**——每个分子是一张图，原子是节点、化学键是边，要预测整个分子的属性。这里用 PyG 的 **TUDataset/PROTEINS** 演示完整的图分类 pipeline：每个图是一种蛋白质的二级结构图，任务是判断它是否是“酶”（二分类）。

相比前面玩具版的 `GraphGIN`（逐图 for 循环训练），这里展示工程上标准的做法：用 `DataLoader` 做 **mini-batch**——把一个 batch 里所有图的邻接拼成一张**断块对角**的大图，用 `batch` 向量记录每个节点属于哪张图，readout 时用 `global_add_pool` 按 `batch` 分组求和。

> 同样需要 `torch_geometric`。当前配套 venv 已经跑通；如果换到新环境，请先安装 PyG，首次运行会自动下载数据集。

In [ ]:
# 需 torch_geometric；首次运行会自动下载数据集。
from torch_geometric.datasets import TUDataset
from torch_geometric.loader import DataLoader

dataset = TUDataset(root='data/TUDataset', name='PROTEINS').shuffle()
print(dataset)                                          # 1113 个图
print(f'类别数 {dataset.num_classes}, 节点特征维数 {dataset.num_node_features}')
print(f'第一个图: {dataset[0]}')

# 80% / 10% / 10% 切分
n = len(dataset)
n_train = int(0.8 * n)
n_val = int(0.1 * n)
train_set = dataset[:n_train]
val_set = dataset[n_train:n_train + n_val]
test_set = dataset[n_train + n_val:]

train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
val_loader = DataLoader(val_set, batch_size=64)
test_loader = DataLoader(test_set, batch_size=64)

# mini-batch：多张图“粘”成一张断块对角的大图
batch = next(iter(train_loader))
print(batch)
print('batch 向量前 20 个:', batch.batch[:20].tolist())

In [ ]:
# 完整的 GIN 图分类器：GINConv + global_add_pool —— 需 torch_geometric
from torch_geometric.nn import GINConv, global_add_pool


class GINGraphClassifier(nn.Module):
    def __init__(self, in_dim, hidden, out_dim, n_layers=3, dropout=0.3):
        super().__init__()
        self.convs = nn.ModuleList()
        for i in range(n_layers):
            d_in = in_dim if i == 0 else hidden
            mlp = nn.Sequential(
                nn.Linear(d_in, hidden), nn.ReLU(),
                nn.Linear(hidden, hidden))
            self.convs.append(GINConv(mlp, train_eps=True))
        # 拼接每层 readout 再分类
        self.classifier = nn.Sequential(
            nn.Linear(hidden * n_layers, hidden),
            nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, out_dim))

    def forward(self, x, edge_index, batch):
        readouts = []
        for conv in self.convs:
            x = F.relu(conv(x, edge_index))
            readouts.append(global_add_pool(x, batch))     # 按图分组求和
        h_graph = torch.cat(readouts, dim=1)
        return self.classifier(h_graph)

In [ ]:
# 训练与评估 —— 需 torch_geometric
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GINGraphClassifier(
    in_dim=dataset.num_node_features,
    hidden=64, out_dim=dataset.num_classes, n_layers=3).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=5e-4)


def evaluate(loader):
    model.eval()
    n_correct = 0
    n_total = 0
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            pred = model(batch.x, batch.edge_index, batch.batch).argmax(dim=1)
            n_correct += (pred == batch.y).sum().item()
            n_total += batch.num_graphs
    return n_correct / n_total


history = {'train_loss': [], 'val_acc': [], 'test_acc': []}
best_val = 0.0
best_state = None
for epoch in range(1, 101):
    model.train()
    ep_loss = 0.0
    ep_n = 0
    for batch in train_loader:
        batch = batch.to(device)
        opt.zero_grad()
        logits = model(batch.x, batch.edge_index, batch.batch)
        loss = F.cross_entropy(logits, batch.y)
        loss.backward()
        opt.step()
        ep_loss += loss.item() * batch.num_graphs
        ep_n += batch.num_graphs
    train_loss = ep_loss / ep_n
    val_acc = evaluate(val_loader)
    test_acc = evaluate(test_loader)
    history['train_loss'].append(train_loss)
    history['val_acc'].append(val_acc)
    history['test_acc'].append(test_acc)
    if val_acc > best_val:
        best_val = val_acc
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
    if epoch % 10 == 0:
        print(f'epoch {epoch:3d}  loss={train_loss:.4f}  '
              f'val={val_acc:.4f}  test={test_acc:.4f}')

model.load_state_dict(best_state)
final_test = evaluate(test_loader)
print(f'\nBest-val 模型 test_acc = {final_test:.4f}')      # 典型 0.74~0.78

In [ ]:
# 结果可视化：混淆矩阵 + ROC 曲线 —— 需 torch_geometric + sklearn
from sklearn.metrics import confusion_matrix, roc_auc_score, roc_curve

model.eval()
all_pred, all_score, all_y = [], [], []
with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)
        logits = model(batch.x, batch.edge_index, batch.batch)
        all_pred.append(logits.argmax(dim=1).cpu())
        all_score.append(F.softmax(logits, dim=1)[:, 1].cpu())
        all_y.append(batch.y.cpu())
y_true = torch.cat(all_y).numpy()
y_pred = torch.cat(all_pred).numpy()
y_score = torch.cat(all_score).numpy()

cm = confusion_matrix(y_true, y_pred)
auc = roc_auc_score(y_true, y_score)
fpr, tpr, _ = roc_curve(y_true, y_score)

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
im = ax[0].imshow(cm, cmap='Blues')
for i in range(2):
    for j in range(2):
        ax[0].text(j, i, cm[i, j], ha='center', va='center',
                   color='white' if cm[i, j] > cm.max() / 2 else 'black')
ax[0].set_xticks([0, 1], ['non-enzyme', 'enzyme'])
ax[0].set_yticks([0, 1], ['non-enzyme', 'enzyme'])
ax[0].set_xlabel('predicted')
ax[0].set_ylabel('true')
ax[0].set_title('混淆矩阵')
plt.colorbar(im, ax=ax[0], fraction=0.04)

ax[1].plot(fpr, tpr, color='#4ECDC4', lw=2, label=f'AUC = {auc:.3f}')
ax[1].plot([0, 1], [0, 1], 'k--', alpha=0.4)
ax[1].set_xlabel('FPR')
ax[1].set_ylabel('TPR')
ax[1].set_title('ROC 曲线')
ax[1].legend()
ax[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

#### 模型预测

训练好的图分类器可以直接对一张新的分子图判断“是不是酶”。取测试集的第一张图试试。

In [2]:
model.eval()
g = test_set[0].to(device)
with torch.no_grad():
    # 单张图，batch 向量全为 0
    batch_vec = torch.zeros(g.num_nodes, dtype=torch.long, device=device)
    pred = model(g.x, g.edge_index, batch_vec).argmax(dim=1).item()
names = ['non-enzyme', 'enzyme']
print(f'预测：{names[pred]}，真实：{names[int(g.y)]}')

预测：non-enzyme，真实：non-enzyme
